# 03. Two Heaps and the Running Median

Use **two heaps together** to split data into a **lower half** and an **upper half**. The median sits at the boundary, available in O(1).

**Learning order:** REAL LIFE → INVARIANT → MEDIAN FINDER → IPO → INTERVIEW THINKING

## 1. Real life

A delivery app wants the **median delivery time** of all orders so far, updated after every order. Sorting after each order is O(n log n) each time. Two heaps give O(log n) per order.

**Setup:**
- `small`: a **max-heap** holding the **lower half**.
- `large`: a **min-heap** holding the **upper half**.

**Invariants:** every element in `small` ≤ every element in `large`, and sizes differ by at most 1 (`small` may have one extra).

**Median:** if sizes are equal → average of the two roots; else the root of the bigger heap.

In [1]:
import heapq

class MedianFinder:
    def __init__(self):
        self.small = []                            # max-heap (store negatives)
        self.large = []                            # min-heap

    def add_num(self, num):
        heapq.heappush(self.small, -num)
        heapq.heappush(self.large, -heapq.heappop(self.small))   # move the largest of small to large
        if len(self.large) > len(self.small):
            heapq.heappush(self.small, -heapq.heappop(self.large))

    def find_median(self):
        if len(self.small) > len(self.large):
            return float(-self.small[0])
        return (-self.small[0] + self.large[0]) / 2

m = MedianFinder()
m.add_num(1); m.add_num(2)
print(m.find_median())   # 1.5
m.add_num(3)
print(m.find_median())   # 2.0

m2 = MedianFinder()
out = []
for x in [5, 15, 1, 3]:
    m2.add_num(x)
    out.append(m2.find_median())
print(out)   # [5.0, 10.0, 5.0, 4.0]

1.5
2.0
[5.0, 10.0, 5.0, 4.0]


## 2. Why push-then-move works

Every new number first goes into `small`, then the **largest** of `small` moves to `large`. This guarantees `small ≤ large` without comparing with both roots. If `large` is now bigger, move its smallest back. Each add = 3 heap operations → O(log n).

**Sliding Window Median** is the hard version: same two heaps plus **lazy deletion** for elements leaving the window.

## 3. IPO (maximize capital)

You can do at most `k` projects. Each needs `capital[i]` to start and gives `profits[i]`. Start with capital `w`. **Greedy:** always pick the most profitable project you can **afford**.

- Sort projects by capital.
- Push affordable projects' profits into a **max-heap**.
- Pop the best, add its profit to `w`, repeat `k` times.

In [2]:
def find_maximized_capital(k, w, profits, capital):
    projects = sorted(zip(capital, profits))
    heap, i = [], 0
    for _ in range(k):
        while i < len(projects) and projects[i][0] <= w:
            heapq.heappush(heap, -projects[i][1])
            i += 1
        if not heap:
            break
        w += -heapq.heappop(heap)
    return w

print(find_maximized_capital(2, 0, [1, 2, 3], [0, 1, 1]))   # 4
print(find_maximized_capital(3, 0, [1, 2, 3], [0, 1, 2]))   # 6

4
6


## 4. Interview thinking

**Signals:** "median of a stream", "balance two halves", "maximize/minimize with two conditions" (affordable + best).

**Say:** "A max-heap holds the smaller half and a min-heap the larger half, sizes balanced. The median is at the roots. Add is O(log n), median O(1)."

**Common mistakes:** forgetting to negate for the max-heap, unbalanced sizes, wrong median for an even count, returning an int instead of a float.

## Quick Revision

- Median = two heaps: max-heap (lower half) + min-heap (upper half).
- Add: push to small, move its max to large, rebalance.
- Median O(1), add O(log n).
- IPO = sort by capital + max-heap of profits among affordable.